In [2]:
import numpy as np
from scipy.interpolate import RegularGridInterpolator

def read_cube_density(filename):
    with open(filename) as f:
        lines = f.readlines()

    title_lines = lines[:2]  # Preserve title/comment lines
    origin_line = lines[2]
    grid_lines = lines[3:6]

    natoms = int(origin_line.split()[0])
    origin = np.array([float(x) for x in origin_line.split()[1:]])

    atom_lines = lines[6:6 + natoms]
    density_lines = lines[6 + natoms:]

    nx, vx = int(grid_lines[0].split()[0]), np.array([float(x) for x in grid_lines[0].split()[1:]])
    ny, vy = int(grid_lines[1].split()[0]), np.array([float(x) for x in grid_lines[1].split()[1:]])
    nz, vz = int(grid_lines[2].split()[0]), np.array([float(x) for x in grid_lines[2].split()[1:]])

    density_flat = np.array([float(val) for line in density_lines for val in line.split()])
    density = density_flat.reshape((nx, ny, nz))

    grid_x = origin[0] + np.arange(nx) * vx[0]
    grid_y = origin[1] + np.arange(ny) * vy[1]
    grid_z = origin[2] + np.arange(nz) * vz[2]

    header = {
        'title': title_lines,
        'origin': origin_line,
        'grid': grid_lines,
        'atoms': atom_lines
    }

    return (grid_x, grid_y, grid_z), density, header, (nx, ny, nz)

# Load both cube files
(grid1, grid2, grid3), rho1, header1, dims1 = read_cube_density("file1.cube")
(gridA, gridB, gridC), rho2, _, _ = read_cube_density("file2.cube")

# Interpolate rho2 onto grid1
interpolator = RegularGridInterpolator((gridA, gridB, gridC), rho2, bounds_error=False, fill_value=0)
mesh_x, mesh_y, mesh_z = np.meshgrid(grid1, grid2, grid3, indexing='ij')
points = np.stack([mesh_x, mesh_y, mesh_z], axis=-1)
rho2_interp = interpolator(points)

# Subtract densities
delta_rho = rho1 - rho2_interp

# Save to new cube file
output_file = "delta_density.cube"
with open(output_file, 'w') as f:
    for line in header1['title']:
        f.write(line)
    f.write(header1['origin'])
    for line in header1['grid']:
        f.write(line)
    for line in header1['atoms']:
        f.write(line)

    nx, ny, nz = dims1
    flat_density = delta_rho.flatten()
    for i in range(0, len(flat_density), 6):
        values = flat_density[i:i+6]
        f.write(" ".join(f"{v:13.5E}" for v in values) + "\n")
